In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('./documents/langchain-history.txt')
doc = loader.load()

In [17]:
# Fixed Sized And Overlap
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=10,  
    separator="",
)

chunks = splitter.split_documents(doc)
print(chunks[0].page_content)
print(chunks[1].page_content)

History of LangChain

LangChain is an open-source framework designed to make it easier for developers to build applications powered by Large Language Models (LLMs). It was created to solve a common pr
common problem: using an LLM directly is simple for basic questions, but building a useful application around an LLM often requires connecting it with external data, tools, memory, APIs, databases, a


In [32]:
# Recursive Chunking (Paragraph - Sentence - Word - Character) 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
)

chunks = splitter.split_documents(doc)
print(len(chunks))
print(chunks[1].page_content)
print(chunks[2].page_content)

31
LangChain is an open-source framework designed to make it easier for developers to build applications powered by Large Language Models (LLMs). It was
(LLMs). It was created to solve a common problem: using an LLM directly is simple for basic questions, but building a useful application around an


In [48]:
# Document Based Structure
from langchain_text_splitters import Language
from langchain_text_splitters import RecursiveCharacterTextSplitter

code = ''' 
import bcrypt
import jwt
from flask import Flask, request, jsonify

app = Flask(__name__)

SECRET_KEY = "my-secret-key"


def generate_token(user):
    """Generate JWT token for the user."""
    return jwt.encode(
        {
            "id": user["id"],
            "email": user["email"],
        },
        SECRET_KEY,
        algorithm="HS256",
    )


def authenticate_user(email, password):
    """Check user credentials."""
    user = find_user_by_email(email)

    if not user:
        return None

    if not bcrypt.checkpw(
        password.encode(),
        user["password"],
    ):
        return None

    return user


@app.route("/login", methods=["POST"])
def login():
    data = request.get_json()

    email = data.get("email")
    password = data.get("password")

    if not email or not password:
        return jsonify({
            "error": "Email and password are required"
        }), 400

    user = authenticate_user(email, password)

    if not user:
        return jsonify({
            "error": "Invalid credentials"
        }), 401

    token = generate_token(user)

    return jsonify({
        "token": token,
        "user": {
            "id": user["id"],
            "email": user["email"],
        }
    }), 200


if __name__ == "__main__":
    app.run(debug=True)
'''

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=275,
    chunk_overlap=0
)

chunks = splitter.split_text(code)

for chunk in chunks:
    print(chunk)
    print("*" * 6)

import bcrypt
import jwt
from flask import Flask, request, jsonify

app = Flask(__name__)

SECRET_KEY = "my-secret-key"
******
def generate_token(user):
    """Generate JWT token for the user."""
    return jwt.encode(
        {
            "id": user["id"],
            "email": user["email"],
        },
        SECRET_KEY,
        algorithm="HS256",
    )
******
def authenticate_user(email, password):
    """Check user credentials."""
    user = find_user_by_email(email)

    if not user:
        return None

    if not bcrypt.checkpw(
        password.encode(),
        user["password"],
    ):
        return None

    return user
******
@app.route("/login", methods=["POST"])
******
def login():
    data = request.get_json()

    email = data.get("email")
    password = data.get("password")

    if not email or not password:
        return jsonify({
            "error": "Email and password are required"
        }), 400
******
user = authenticate_user(email, password)

    if not user:

In [57]:
# Sementic Meaning Based Chunking
from langchain_experimental.text_splitter import SemanticChunker
from langchain_mistralai import MistralAIEmbeddings

text = ''' 
The farmer woke up early in the morning and walked through his fields to check the condition of his crops. The soil was dry after several weeks without rain, so he decided to start the irrigation pump. He was also planning to buy better quality seeds for the next planting season. Rising fertilizer prices were becoming a major concern, and he was looking for ways to reduce farming costs without affecting crop production.

The cricket match between India and Australia was extremely exciting. India needed 12 runs from the final over, and the batsman hit a boundary on the first ball. The bowler then delivered two excellent balls and stopped the scoring. With only two runs needed from the final ball, the batsman hit the ball over the boundary and won the match for his team. Thousands of fans celebrated the victory in the stadium.

The government announced a new policy focused on improving public transportation in major cities. The proposal includes expanding metro networks, improving bus services, and introducing new electric buses. Political leaders have different opinions about the plan, with some supporting the investment while others are concerned about the cost. The policy is expected to become an important topic during the upcoming election campaign.

A software company recently released a new smartphone application for managing personal finances. The application allows users to track expenses, create monthly budgets, and receive notifications when they are spending too much. It also provides charts that show how money is distributed across categories such as food, transportation, entertainment, and shopping.

Mount Everest is the highest mountain above sea level, reaching an elevation of approximately 8,849 meters. It is located in the Himalayas on the border between Nepal and China. Thousands of climbers have attempted to reach its summit, but the extreme altitude, freezing temperatures, and unpredictable weather make the journey extremely dangerous.
'''

embeddings = MistralAIEmbeddings()

splitter = SemanticChunker(embeddings,breakpoint_threshold_type='standard_deviation',breakpoint_threshold_amount=1)

docs = splitter.create_documents([text])

for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)

--- Chunk 1 ---
 
The farmer woke up early in the morning and walked through his fields to check the condition of his crops. The soil was dry after several weeks without rain, so he decided to start the irrigation pump. He was also planning to buy better quality seeds for the next planting season.
--- Chunk 2 ---
Rising fertilizer prices were becoming a major concern, and he was looking for ways to reduce farming costs without affecting crop production. The cricket match between India and Australia was extremely exciting. India needed 12 runs from the final over, and the batsman hit a boundary on the first ball. The bowler then delivered two excellent balls and stopped the scoring. With only two runs needed from the final ball, the batsman hit the ball over the boundary and won the match for his team. Thousands of fans celebrated the victory in the stadium.
--- Chunk 3 ---
The government announced a new policy focused on improving public transportation in major cities. The proposal inc